# Swin-Tiny Results Summary

This notebook summarises the Swin-Tiny experiment for the CSC3109 aerial scene classification project.

Model: `swin_tiny_patch4_window7_224` with pretrained ImageNet weights.

Classes: `bridge`, `freeway`, `overpass`, `railway`.

## Experiment protocol

- Training images: `data/train` with 700 images per class.
- Internal tuning split: 80/20 stratified split from `data/train`.
- Held-out validation set: `data/val`, 100 images per class.
- The held-out validation set was not used during training.
- Training stopped early at epoch 6 because tuning macro-F1 stopped improving.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate project root. Run this notebook from the project or notebooks folder.")

PROJECT_ROOT = find_project_root()
HISTORY_PATH = PROJECT_ROOT / "model" / "swin_tiny" / "history.csv"
TUNE_METRICS_PATH = PROJECT_ROOT / "model" / "swin_tiny" / "best_tune_metrics.json"
HELDOUT_METRICS_PATH = PROJECT_ROOT / "reports" / "swin_tiny_eval" / "metrics.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"History exists: {HISTORY_PATH.exists()}")
print(f"Tune metrics exists: {TUNE_METRICS_PATH.exists()}")
print(f"Held-out metrics exists: {HELDOUT_METRICS_PATH.exists()}")

In [ ]:
history = pd.read_csv(HISTORY_PATH)
tune_metrics = json.loads(TUNE_METRICS_PATH.read_text())
heldout_metrics = json.loads(HELDOUT_METRICS_PATH.read_text())

history

## Key metrics

## Direct held-out validation test

This section loads the saved Swin-Tiny checkpoint and directly runs inference on `data/val`. This confirms the model performance from the notebook itself rather than only reading the saved metrics file.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from tqdm.auto import tqdm

from src.config import IMAGE_SIZE, VAL_DIR
from src.data import build_eval_transform
from src.evaluation import classification_metrics
from src.models import build_swin_classifier

CHECKPOINT_PATH = PROJECT_ROOT / "model" / "swin_tiny" / "best_model.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
class_to_idx = checkpoint["class_to_idx"]
class_names = [name for name, _ in sorted(class_to_idx.items(), key=lambda item: item[1])]
variant = checkpoint.get("args", {}).get("variant", "tiny")

val_dataset = datasets.ImageFolder(VAL_DIR, transform=build_eval_transform(IMAGE_SIZE))
assert val_dataset.class_to_idx == class_to_idx, (val_dataset.class_to_idx, class_to_idx)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
model = build_swin_classifier(num_classes=len(class_names), variant=variant, pretrained=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device).eval()

y_true = []
y_pred = []
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Testing on data/val"):
        logits = model(images.to(device))
        predictions = logits.argmax(dim=1).cpu().tolist()
        y_true.extend(labels.tolist())
        y_pred.extend(predictions)

notebook_val_metrics = classification_metrics(y_true, y_pred, class_names)
print(f"Device: {device}")
print(f"Validation images tested: {len(y_true)}")
print(f"Accuracy: {notebook_val_metrics['accuracy']:.4f}")
print(f"Macro F1: {notebook_val_metrics['macro_f1']:.4f}")

In [ ]:
pd.DataFrame({name: notebook_val_metrics["classification_report"][name] for name in class_names}).T

In [ ]:
summary = pd.DataFrame([
    {
        "split": "internal tune",
        "accuracy": tune_metrics["accuracy"],
        "macro_precision": tune_metrics["macro_precision"],
        "macro_recall": tune_metrics["macro_recall"],
        "macro_f1": tune_metrics["macro_f1"],
    },
    {
        "split": "held-out validation",
        "accuracy": notebook_val_metrics["accuracy"],
        "macro_precision": notebook_val_metrics["macro_precision"],
        "macro_recall": notebook_val_metrics["macro_recall"],
        "macro_f1": notebook_val_metrics["macro_f1"],
    },
])

assert abs(notebook_val_metrics["accuracy"] - heldout_metrics["accuracy"]) < 1e-12
assert abs(notebook_val_metrics["macro_f1"] - heldout_metrics["macro_f1"]) < 1e-12
summary

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train loss")
axes[0].plot(history["epoch"], history["tune_loss"], marker="o", label="tune loss")
axes[0].set_title("Loss by epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()

axes[1].plot(history["epoch"], history["train_accuracy"], marker="o", label="train accuracy")
axes[1].plot(history["epoch"], history["tune_accuracy"], marker="o", label="tune accuracy")
axes[1].plot(history["epoch"], history["tune_macro_f1"], marker="o", label="tune macro-F1")
axes[1].set_title("Accuracy / F1 by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0.94, 1.01)
axes[1].legend()

fig.tight_layout()
plt.show()

## Held-out validation confusion matrix

In [ ]:
class_names = ["bridge", "freeway", "overpass", "railway"]
confusion = pd.DataFrame(
    notebook_val_metrics["confusion_matrix"],
    index=[f"true {name}" for name in class_names],
    columns=[f"pred {name}" for name in class_names],
)

plt.figure(figsize=(7, 5))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues")
plt.title("Swin-Tiny held-out validation confusion matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.tight_layout()
plt.show()

confusion

## Per-class held-out validation metrics

In [ ]:
report = notebook_val_metrics["classification_report"]
per_class = pd.DataFrame({name: report[name] for name in class_names}).T
per_class

## Interpretation

The pretrained Swin-Tiny model achieved very high held-out validation performance:

- Accuracy: **99.75%**
- Macro-F1: **99.75%**
- Correct predictions: **399 / 400**
- Error pattern: **1 bridge image was classified as overpass**

This error is plausible because bridges and overpasses share similar aerial visual structures: elevated road segments, shadows, and linear transport corridors. The model correctly classified every freeway, overpass, and railway sample in the held-out split.

The result suggests that Swin-Tiny is a strong final-model candidate for this dataset. Its shifted-window attention is suitable for combining local texture cues, such as rails and road lanes, with larger spatial layout cues, such as crossings and elevated structures.

## Report-ready summary

> We fine-tuned a pretrained Swin-Tiny Transformer on the 4-class aerial scene classification task. The model was trained using an internal 80/20 split from the training set, while the official held-out validation set was reserved for final evaluation. On the held-out validation set of 400 images, the model achieved 99.75% accuracy and 99.75% macro-F1. The confusion matrix showed only one error, where a bridge image was predicted as overpass, which is semantically reasonable due to the strong visual similarity between elevated transport structures.